# Kaggle Training V3.0 (Stable & Optimized)\n
This updated V3.0 focuses on **Absolute Stability** on Kaggle CPUs/GPUs while retaining critical medical AI optimizations.\n
\n
**Improvements over unstable V3 scripts:**\n
- **Fast Augmentations:** Reverted from `albumentations` back to base `torchvision.transforms` to dramatically reduce CPU bottlenecking.\n
- **Removed EMA Models & MixUp:** Prevents duplicate models from consuming all 16GB of VRAM causing Kaggle crashes.\n
- **Kept Asymmetric Loss:** Heavily penalizes False Negatives for critical diseases like Melanoma.\n
- **Kept Square-Root Sampler:** Dynamically balances massive class differences (e.g. 5000 Nevus vs 150 Vascular Lesions) without oversampling causing collapse.\n
- **Clean Logging:** Removed inner-loop `tqdm` to prevent browser tab from hanging with 100,000+ lines of output.

In [ ]:

import os
import glob
import random
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from tqdm.auto import tqdm
import torch.nn.functional as F
from sklearn.metrics import f1_score

# --- 1. SETTINGS & REPRODUCIBILITY ---
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

# --- 2. ROBUST DATASET DISCOVERY ---
EXTRACT_DIR = None
PRETRAINED_MODEL = None

known_paths = [
    "/kaggle/input/derma-db-24class/processed",
    "/kaggle/input/dermadb-24class-v2/processed",
    "/kaggle/input/datasets/dnghongkhang/derma-db-24class/processed",
    "/kaggle/input/dermafusion-processed/processed"
]

for path in known_paths:
    if os.path.exists(os.path.join(path, "train")):
        EXTRACT_DIR = path
        break

if not EXTRACT_DIR and os.path.exists("/kaggle/input"):
    import itertools
    for root, dirs, files in itertools.islice(os.walk("/kaggle/input"), 200):
        if "train" in dirs and "val" in dirs:
            EXTRACT_DIR = root
            break

if not EXTRACT_DIR:
    raise RuntimeError("Dataset NOT FOUND. Add the derma-db-24class dataset to this Kaggle notebook.")
else:
    print(f"Dataset path: {EXTRACT_DIR}")

known_model_paths = [
    "/kaggle/input/derma-v21-weights/efficientnet_b4_derma_v2_1_finetuned.pth",
]
for kp in known_model_paths:
    if os.path.exists(kp):
        PRETRAINED_MODEL = kp
        break
if not PRETRAINED_MODEL:
    model_paths = glob.glob("/kaggle/input/**/*.pth", recursive=True)
    for mp in model_paths:
        if "efficientnet_b4" in os.path.basename(mp):
            PRETRAINED_MODEL = mp
            break

if PRETRAINED_MODEL:
    print(f"Pretrained model found: {PRETRAINED_MODEL}")
else:
    print("No pretrained model found. Training from ImageNet weights.")

# --- 3. FAST AUGMENTATIONS ---
IMG_SIZE = 380
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 20),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# --- 4. DATA LOADING & SAMPLING ---
BATCH_SIZE = 48
NUM_WORKERS = 4

train_dir = os.path.join(EXTRACT_DIR, "train")
val_dir = os.path.join(EXTRACT_DIR, "val")

train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)
class_names = train_ds.classes
NUM_CLASSES = len(class_names)
print(f"Found {NUM_CLASSES} classes.")

labels = [y for _, y in train_ds.samples]
counts = np.bincount(labels, minlength=NUM_CLASSES)

class_weights = 1.0 / np.sqrt(counts + 1e-6)
class_weights_tensor = torch.tensor(class_weights / (class_weights.sum() / NUM_CLASSES), dtype=torch.float32, device=device)

sample_weights = [class_weights[label] for label in labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    persistent_workers=(NUM_WORKERS > 0)
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# --- 5. MODEL INITIALIZATION ---
model = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=False),
    nn.Linear(in_features, NUM_CLASSES)
)

if PRETRAINED_MODEL:
    print(f"Loading pre-trained weights from {PRETRAINED_MODEL}")
    ckpt = torch.load(PRETRAINED_MODEL, map_location=device, weights_only=False)
    if "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"])
    elif "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)

model = model.to(device)

# --- 6. ASYMMETRIC LOSS (Fixed modulation logic) ---
class SimpleAsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        targets_oh = F.one_hot(targets, num_classes=num_classes).float()
        probs_pos = torch.softmax(logits, dim=1)
        probs_neg = (1 - probs_pos + self.clip).clamp(max=1)

        log_pos = torch.log(probs_pos.clamp(min=1e-8))
        log_neg = torch.log(probs_neg.clamp(min=1e-8))

        loss = (
            -targets_oh * (1 - probs_pos) ** self.gamma_pos * log_pos
            - (1 - targets_oh) * probs_pos ** self.gamma_neg * log_neg
        )
        return loss.sum(dim=1).mean()

criterion = SimpleAsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0)

optimizer = torch.optim.AdamW([
    {"params": model.features.parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 5e-5}
], weight_decay=1e-3)

NUM_EPOCHS = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=15, T_mult=1, eta_min=1e-7
)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

# --- 7. STABLE TRAINING LOOP ---
best_val_f1 = 0.0
PATIENCE = 10
epochs_no_improve = 0
history = []

print("Starting V3.0 training...")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)

        # Step scheduler per batch (required for CosineAnnealingWarmRestarts)
        scheduler.step(epoch - 1 + batch_idx / len(train_loader))

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(images)
                loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            preds = logits.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    val_acc = (np.array(all_preds) == np.array(all_labels)).mean()

    history.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_f1": round(val_f1, 4),
        "val_acc": round(val_acc, 4)
    })

    print(f"Epoch {epoch}/{NUM_EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | val_f1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        tmp_path = "efficientnet_b4_derma_v3_0.pth.tmp"
        torch.save({"model_state": model.state_dict(), "val_f1": best_val_f1, "epoch": epoch}, tmp_path)
        os.replace(tmp_path, "efficientnet_b4_derma_v3_0.pth")
        print(f"  Saved best checkpoint (F1: {best_val_f1:.4f})")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print("Early stopping triggered.")
        break

# Restore best model weights at end of training
best_ckpt = torch.load("efficientnet_b4_derma_v3_0.pth", map_location=device, weights_only=False)
model.load_state_dict(best_ckpt["model_state"])
print(f"Restored best model from epoch {best_ckpt['epoch']} (F1: {best_ckpt['val_f1']:.4f})")

with open("training_history_v3.json", "w") as f:
    json.dump(history, f, indent=4)

print("Training complete.")




## Evaluation & Visualization
Plot Learning Curves and Confusion Matrix after training is complete.

In [ ]:
# ============================================================
# STANDALONE EVALUATION CELL — Thesis Report Visualizations
# Run this cell independently. Attach training output as dataset.
# Saves all plots to /kaggle/working/ for download.
# ============================================================
import os, glob, json
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize

IMG_SIZE   = 380
BATCH_SIZE = 48
NUM_WORKERS = 4
OUTPUT_DIR  = "/kaggle/working"
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- Locate Dataset ---
EXTRACT_DIR = None
for p in ["/kaggle/input/derma-db-24class/processed",
          "/kaggle/input/datasets/khang2222/derma-db-24class/processed"]:
    if os.path.exists(os.path.join(p, "val")):
        EXTRACT_DIR = p
        break
if not EXTRACT_DIR:
    import itertools
    for root, dirs, _ in itertools.islice(os.walk("/kaggle/input"), 200):
        if "val" in dirs:
            EXTRACT_DIR = root
            break
if not EXTRACT_DIR:
    raise RuntimeError("Validation dataset not found.")
print(f"Dataset: {EXTRACT_DIR}")

# --- Locate Checkpoint ---
CHECKPOINT = None
for p in ["/kaggle/working/efficientnet_b4_derma_v3_0.pth",
          "/kaggle/input/derma-v3-checkpoint/efficientnet_b4_derma_v3_0.pth"]:
    if os.path.exists(p):
        CHECKPOINT = p
        break
if not CHECKPOINT:
    for c in glob.glob("/kaggle/input/**/*.pth", recursive=True):
        if "efficientnet_b4" in os.path.basename(c):
            CHECKPOINT = c
            break
if not CHECKPOINT:
    raise RuntimeError("Checkpoint not found.")
print(f"Checkpoint: {CHECKPOINT}")

# --- Val Loader ---
val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 20), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD),
])
val_ds     = datasets.ImageFolder(os.path.join(EXTRACT_DIR, "val"), transform=val_tfms)
class_names = val_ds.classes
NUM_CLASSES = len(class_names)
val_loader  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         prefetch_factor=2 if NUM_WORKERS > 0 else None,
                         persistent_workers=(NUM_WORKERS > 0))
print(f"Val: {len(val_ds)} images | {NUM_CLASSES} classes | {len(val_loader)} batches")

# --- Load Model ---
model = efficientnet_b4(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=False),
    nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
)
ckpt  = torch.load(CHECKPOINT, map_location=device, weights_only=False)
state = ckpt.get("model_state", ckpt.get("model_state_dict", ckpt))
model.load_state_dict(state)
model = model.to(device)
model.eval()
print(f"Loaded epoch={ckpt.get('epoch','?')} | best F1={ckpt.get('val_f1','?')}")

# --- Run Inference ---
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(np.argmax(probs, axis=1))
        all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

# ================================================================
# FIGURE 1: Learning Curves (Loss + Accuracy + F1)
# ================================================================
hist_path = "/kaggle/working/training_history_v3.json"
if not os.path.exists(hist_path):
    hist_path = os.path.join(os.path.dirname(CHECKPOINT), "training_history_v3.json")

if os.path.exists(hist_path):
    with open(hist_path) as f:
        history = json.load(f)
    epochs     = [h["epoch"]           for h in history]
    train_loss = [h["train_loss"]       for h in history]
    val_loss   = [h["val_loss"]         for h in history]
    val_f1     = [h.get("val_f1",  0)  for h in history]
    val_acc    = [h.get("val_acc", 0)  for h in history]
    best_epoch = max(history, key=lambda h: h.get("val_f1", 0))["epoch"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Learning Curves — EfficientNet-B4 V3.0", fontsize=16, y=1.02)

    axes[0].plot(epochs, train_loss, 'b-o', lw=2, ms=4, label='Train Loss')
    axes[0].plot(epochs, val_loss,   'r-o', lw=2, ms=4, label='Val Loss')
    axes[0].axvline(best_epoch, color='green', ls='--', lw=1.2, label=f'Best (Ep {best_epoch})')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, ls='--', alpha=0.5)

    axes[1].plot(epochs, val_acc, 'g-o', lw=2, ms=4, label='Val Accuracy')
    axes[1].axvline(best_epoch, color='green', ls='--', lw=1.2)
    axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, ls='--', alpha=0.5)

    axes[2].plot(epochs, val_f1, color='purple', marker='o', lw=2, ms=4, label='Val Macro F1')
    axes[2].axvline(best_epoch, color='green', ls='--', lw=1.2)
    axes[2].set_title('Validation Macro F1'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1')
    axes[2].legend(); axes[2].grid(True, ls='--', alpha=0.5)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig1_learning_curves.png")
    plt.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.3)
    plt.close()
    print(f"Saved: {path}")
else:
    print("training_history_v3.json not found — skipping Figure 1.")

# ================================================================
# FIGURE 2: Confusion Matrix (count + row %)
# ================================================================
cm      = confusion_matrix(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Validation Set V3.0', fontsize=15, pad=20)
plt.xlabel('Predicted Label', fontsize=12, labelpad=10)
plt.ylabel('True Label', fontsize=12, labelpad=10)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "fig2_confusion_matrix.png")
plt.savefig(path, dpi=200, bbox_inches='tight', pad_inches=0.5)
plt.close()
print(f"Saved: {path}")

ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=90, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Per-Class Precision / Recall / F1 — Validation Set V3.0', fontsize=15)
ax.axhline(0.9, color='red', ls='--', lw=1.2, label='0.90 threshold')
ax.legend(fontsize=12)
ax.grid(axis='y', ls='--', alpha=0.5)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "fig3_per_class_metrics.png")
plt.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.3)
plt.close()
print(f"Saved: {path}")

# ================================================================
# FIGURE 4: Macro-averaged ROC Curve (One-vs-Rest)
# ================================================================
labels_bin = label_binarize(all_labels, classes=np.arange(NUM_CLASSES))
fpr_macro = np.linspace(0, 1, 300)
tprs = []
fig, ax = plt.subplots(figsize=(8, 7))
for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    tprs.append(np.interp(fpr_macro, fpr, tpr))
    ax.plot(fpr, tpr, lw=0.9, alpha=0.45, label=f'{cls} (AUC={roc_auc:.2f})')

mean_tpr = np.mean(tprs, axis=0)
mean_auc = auc(fpr_macro, mean_tpr)
ax.plot(fpr_macro, mean_tpr, 'k-', lw=3, label=f'Macro avg (AUC={mean_auc:.3f})')
ax.plot([0,1],[0,1],'r--', lw=1.2, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate',  fontsize=13)
ax.set_title('ROC Curves — One-vs-Rest (All Classes + Macro Average)', fontsize=13)
ax.legend(fontsize=7, loc='lower right', ncol=2)
ax.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "fig4_roc_curves.png")
plt.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.3)
plt.close()
print(f"Saved: {path}")

# ================================================================
# FIGURE 5: Precision-Recall Curve (Macro)
# ================================================================
fig, ax = plt.subplots(figsize=(8, 7))
for i, cls in enumerate(class_names):
    prec_c, rec_c, _ = precision_recall_curve(labels_bin[:, i], all_probs[:, i])
    ap = average_precision_score(labels_bin[:, i], all_probs[:, i])
    ax.plot(rec_c, prec_c, lw=0.9, alpha=0.45, label=f'{cls} (AP={ap:.2f})')

macro_ap = average_precision_score(labels_bin, all_probs, average='macro')
ax.axhline(macro_ap, color='k', lw=2.5, ls='--', label=f'Macro AP = {macro_ap:.3f}')
ax.set_xlabel('Recall',    fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curves — All Classes + Macro AP', fontsize=13)
ax.legend(fontsize=7, loc='upper right', ncol=2)
ax.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "fig5_precision_recall_curves.png")
plt.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.3)
plt.close()
print(f"Saved: {path}")

# ================================================================
# FIGURE 6: Class Distribution — Train vs Val
# ================================================================
train_ds = datasets.ImageFolder(os.path.join(EXTRACT_DIR, "train"))
t_counts  = np.bincount([y for _, y in train_ds.samples], minlength=NUM_CLASSES)
v_counts  = np.bincount(all_labels, minlength=NUM_CLASSES)

x  = np.arange(NUM_CLASSES)
fig, ax = plt.subplots(figsize=(22, 7))
ax.bar(x - 0.35, t_counts, 0.35, label='Train', color='steelblue', alpha=0.85)
ax.bar(x,        v_counts, 0.35, label='Val',   color='darkorange',  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=90, fontsize=10)
ax.set_ylabel('Number of Images', fontsize=13)
ax.set_title('Class Distribution — Train vs Val Sets V3.0', fontsize=15)
ax.legend(fontsize=12)
ax.grid(axis='y', ls='--', alpha=0.5)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "fig6_class_distribution.png")
plt.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.3)
plt.close()
print(f"Saved: {path}")

# ================================================================
# TEXT: Full Classification Report
# ================================================================
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))
print("\nAll figures saved to /kaggle/working/")

